# Particle tracking and the water table

**Under the table and dropping**, a soliloquy

```
To drape or to drop, that’s the question—
Whether ’tis nobler in the porous mind
To suffer the gradients of outrageous flux,
Or to take arms against a sea of heads
And, by opposing, end them.
To flow, to seep—
No more—and by a seep to say we end
The slings and arrows of stochastic rain,
The thousand natural shocks that aquifers bear—
’Tis a consummation devoutly sensed.
To flow, to seep—
To seep, perchance to plume—ay, there’s the rub:
For in that plume of dissolved mystery
What heterogeneity may come,
When we have shuffled off this Darcy coil,
Must give us pause.
There’s the respect
That makes calamity of long retardation:
For who would bear the whips and scorns of time—
Pumping stress, inverse woes, parameter shame—
The pangs of unmodeled boundaries, the law’s delay,
The insolence of numerical dispersion,
And spurns that patient merit of the grid—
When he himself might his quietus make
With but a steady state?
Who would burdens bear,
To grunt and sweat beneath a transient load,
But that the dread of something after flow—
The undiscovered zone from which no tracer
Returns unbiased—puzzles the will,
And makes us rather bear those ills we have
Than drift to others that we know not of?
Thus conscience does make cowards of us all,
And thus the native hue of resolution
Is sicklied o’er with the pale cast of doubt,
And enterprises of great pitch and moment—
Conceptual models, grand and elegant—
With this regard their currents turn awry
And lose the name of action.
Soft you now,
Fair aquifer—remember me in recharge.
```

&mdash; some LLM, doing what it does best

## What this notebook covers

Build a MODFLOW 6 **groundwater flow (GWF)** model of a transient seepage scenario, then build a **particle-tracking (PRT)** model that reuses the flow model's saved output to trace where water goes. By the end you will be able to run a flow model, feed its heads and cell budgets to a PRT model, and plot particle pathlines in map view and in 3D.

The scenario is a transient flow system whose upper layers start out dry and gradually wet up over the simulation. Particles are released into the dry top layer at the start of the simulation. By default MODFLOW makes dry cells inactive, and particles cannot be tracked through inactive regions. Enabling the **Newton** formulation instead keeps dry cells active, so tracking can proceed. Run the flow model first without Newton, then with Newton, and see some of the configurable behaviors PRT supports for particles in dry conditions.

Reference docs: the [MODFLOW 6 input/output guide](https://modflow6.readthedocs.io/en/latest/mf6io.html) and the [FloPy documentation](https://flopy.readthedocs.io/).

#### Define the units

MODFLOW 6 has no built-in unit system; it only requires that every input share a consistent length and time unit. Use meters and days throughout this notebook, and set `time_unit` and `length_unit` so the discretization packages record them.

In [ ]:
time_unit = "days"
length_unit = "meters"

#### Set up the temporal discretization

Divide the simulation into 3 **stress periods** — blocks of time over which the boundary conditions are held constant: a steady-state period, a transient period, and a final steady-state period. The injection well is active in the transient period. Each row of `tdis_pd` is one period, given as a tuple:

```python
# (period length, number of time steps, time-step multiplier)
(300, 30, 1.1)
```

so the middle period runs 300 days, split into 30 time steps whose length grows by a factor of 1.1 each.

In [ ]:
nper = 3
tdis_pd = [
    (1, 1, 1.0),
    (300, 30, 1.1),
    (1000, 1, 1.0),
]

#### Set up the spatial discretization

Build a grid of 3 layers, 20 rows, and 20 columns. Each layer is 10 m thick below a top surface at 1600 m, so compute each layer bottom (`bot`) by subtracting the cumulative thickness from `top`. Cell IDs throughout are zero-based `(layer, row, column)` tuples counting from 0.

In [ ]:
import numpy as np

nlay, nrow, ncol = 3, 20, 20
Lx, Ly = 100.0, 100.0
delr, delc = Lx / ncol, Ly / nrow
top = 1600.0
thickness = 10.0
bot = np.zeros((nlay, nrow, ncol))
for k in range(nlay):
    bot[k] = top - (k + 1) * thickness

#### Set up the workspaces

Create a base workspace for the whole example and a `gwf` subfolder for the flow model. `pathlib.Path` builds OS-independent paths, and `mkdir(parents=True)` creates the folders. Keeping the flow and particle-tracking models in separate folders matters later, when PRT reads the flow model's output files by relative path.

In [ ]:
from pathlib import Path

example_name = "prt-wt"
base_ws = Path("models") / example_name
gwf_name = f"{example_name}-gwf"
gwf_ws = base_ws / "gwf"
gwf_ws.mkdir(exist_ok=True, parents=True)

## Phase 1 — Build the flow model

Everything up to the dry-cell check builds and runs the GWF model. Its saved heads and cell budgets become the input to the PRT model in Phase 2.

Create the simulation, temporal discretization, model, spatial discretization, and initial conditions in one cell with `flopy.mf6.MFSimulation()`, `ModflowTdis()`, `ModflowGwf()`, `ModflowGwfdis()`, and `ModflowGwfic()`. Set the starting head (`strt`) to the top of the model. The `# options="NEWTON"` line is commented out for this first pass; uncomment it on the second pass to keep dry cells active.

In [ ]:
import flopy

gwf_sim = flopy.mf6.MFSimulation(
    sim_name=gwf_name,
    exe_name="mf6",
    version="mf6",
    sim_ws=gwf_ws,
)
gwf_tdis = flopy.mf6.ModflowTdis(
    gwf_sim,
    time_units=time_unit,
    nper=nper,
    perioddata=tdis_pd,
)
gwf = flopy.mf6.ModflowGwf(
    gwf_sim,
    modelname=gwf_name,
    # options="NEWTON",  # TODO uncomment after first pass
    save_flows=True,
)
gwf_dis = flopy.mf6.ModflowGwfdis(
    gwf,
    length_units=length_unit,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=delr,
    delc=delc,
    top=top,
    botm=bot,
)
gwf_ic = flopy.mf6.ModflowGwfic(gwf, strt=top)

#### Node Property Flow (NPF) package

Set the aquifer's hydraulic conductivity with `flopy.mf6.ModflowGwfnpf()`. `icelltype=1` makes every cell **convertible** — able to switch between confined and unconfined as it wets and dries — which is what lets the upper layers go dry. Horizontal conductivity `k=0.5` exceeds vertical `k33=0.1`, so water moves more easily laterally than vertically. Save specific discharge and saturation for later use.

In [ ]:
npf = flopy.mf6.ModflowGwfnpf(
    gwf,
    icelltype=1,
    k=0.5,
    k33=0.1,
    save_specific_discharge=True,
    save_saturation=True,
    save_flows=True,
)

#### Storage (STO) package

Add aquifer storage with `flopy.mf6.ModflowGwfsto()` so the transient period can fill and drain. Match `iconvert=1` to the NPF cell type, set specific storage `ss` and specific yield `sy`, and mark which periods are steady-state versus transient using dicts keyed by the zero-based period number — periods 0 and 2 steady, period 1 transient.

In [ ]:
sto = flopy.mf6.ModflowGwfsto(
    gwf,
    iconvert=1,
    ss=0.0001,
    sy=0.1,
    steady_state={0: True, 2: True},
    transient={1: True},
    save_flows=True,
)

#### Constant-head (CHD) boundaries

Pin the head at two cells with `flopy.mf6.ModflowGwfchd()` to drive a gentle gradient across the grid, from 1587 m in the upper-left to 1582 m in the lower-right. `CHD` is a list-based stress package: its `stress_period_data` is a dict keyed by the zero-based stress period, and each entry is a tuple of

```python
# (layer, row, column, head)
(1, 0, 0, 1587.0)
```

The same two cells apply in every period, so the dict comprehension repeats them for periods 0 through 2.

In [ ]:
chd0_cellid = (1, 0, 0)
chd1_cellid = (1, nrow - 1, ncol - 1)
chd = flopy.mf6.ModflowGwfchd(
    gwf,
    stress_period_data={
        i: [(*chd0_cellid, 1587.0), (*chd1_cellid, 1582.0)] for i in range(nper)
    },
    save_flows=True,
)

#### Well (WEL) package

Add an injection well in the middle layer of the upper-left quadrant with `flopy.mf6.ModflowGwfwel()`. Each `WEL` entry is a tuple of

```python
# (layer, row, column, rate)
(1, 5, 5, 100.0)
```

A positive rate injects water, here at 100 m³/d. The dict comprehension over `range(1, nper)` activates the well from the second stress period onward.

In [ ]:
wel_cellid = (1, nrow // 4, ncol // 4)
well = flopy.mf6.ModflowGwfwel(
    gwf,
    stress_period_data={i: [(*wel_cellid, 100.0)] for i in range(1, nper)},
    save_flows=True,
)

#### Output Control (OC) package

Tell MODFLOW which results to write and where with `flopy.mf6.ModflowGwfoc()`. Save heads to a `.hds` file and cell-by-cell flows to a `.cbb` file for every time step. The PRT model reads exactly these two files, so both must be saved.

In [ ]:
oc = flopy.mf6.ModflowGwfoc(
    gwf,
    budget_filerecord=f"{gwf_name}.cbb",
    head_filerecord=f"{gwf_name}.hds",
    saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
)

#### Iterative Model Solution (IMS)

Configure the solver with `flopy.mf6.ModflowIms()`. For the standard formulation, `complexity="MODERATE"` and the head-change closure tolerances below are enough. The commented block in the next cell is a more robust configuration for when Newton is enabled — Newton keeps dry cells active but is harder to converge, so it needs tighter under-relaxation and a different linear accelerator.

In [ ]:
ims = flopy.mf6.ModflowIms(
    gwf_sim,
    complexity="MODERATE",
    outer_dvclose=1e-5,
    inner_dvclose=1e-6,
    pname=gwf_name,
)

In [ ]:
# TODO: Comment out the block above and uncomment this one when Newton is enabled.

# flopy.mf6.ModflowIms(
#     gwf_sim,
#     print_option="SUMMARY",
#     outer_dvclose=1e-5,
#     outer_maximum=100,
#     under_relaxation="DBD",
#     under_relaxation_gamma=0.01,
#     under_relaxation_theta=0.7,
#     under_relaxation_kappa=0.01,
#     under_relaxation_momentum=0.0,
#     inner_maximum=100,
#     inner_dvclose=1e-6,
#     rcloserecord=0.1,
#     linear_acceleration="BICGSTAB",
#     relaxation_factor=0.99,
#     number_orthogonalizations=2,
#     reordering_method="NONE",
#     pname=gwf_name,
# )

#### Write and run the flow model

Write the MODFLOW 6 input files with `gwf_sim.write_simulation()` and run them with `.run_simulation()`. The `assert` stops the notebook if MODFLOW does not finish normally.

In [ ]:
gwf_sim.write_simulation(silent=True)
success, buff = gwf_sim.run_simulation(silent=False)
assert success, "MODFLOW 6 did not terminate normally"

#### Load and plot the heads

Load the head output with `gwf.output.head()`, then pull the array at the end of each stress period with `.get_data(kstpkper=...)`, where `kstpkper` is a zero-based `(time step, stress period)` pair. Plot the middle-layer head at the end of each period.

The **water table** is the top of the saturated zone. During the initial steady-state period the constant-head boundaries drain the system until the top layer sits entirely above the water table and goes dry. Over the transient period the injection well and boundaries raise the water table until the top layer is partially saturated again. This rise and fall is exactly what the PRT model must cope with when it releases particles into the top layer.

In [ ]:
hds = gwf.output.head()
hds_kper0 = hds.get_data(kstpkper=(0, 0))
hds_kper1 = hds.get_data(kstpkper=(0, 1))
hds_kper2 = hds.get_data(kstpkper=(0, 2))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

with flopy.plot.styles.USGSPlot():
    fig, axes = plt.subplots(1, 3, figsize=(11, 5))
    for i, (ax, arr, title) in enumerate(
        [
            (axes[0], hds_kper0[1], "Layer 1 head, end of period 1"),
            (axes[1], hds_kper1[1], "Layer 1 head, end of period 2"),
            (axes[2], hds_kper2[1], "Layer 1 head, end of period 3"),
        ]
    ):
        ax.set_aspect("equal")
        ax.set_title(title, fontsize=10)
        if i == 0:
            ax.legend(
                handles=[
                    Patch(color="red", label="WELL"),
                    Patch(color="blue", label="CHD"),
                ],
            )
        mm = flopy.plot.PlotMapView(gwf, ax=ax, layer=1)
        mm.plot_grid(alpha=0.2)
        pc = mm.plot_array(arr, alpha=0.5, vmin=1582, vmax=1587)
        plt.colorbar(pc, ax=ax, shrink=0.25, label="Head (m)")
        mm.plot_bc("WELL", plotAll=True, kper=1, color="red")
        mm.plot_bc("CHD", plotAll=True, color="steelblue")

    fig.tight_layout()
    plt.show()

**What to look for.** Head is highest near the upper-left constant-head cell (blue) and lowest near the lower-right one, so water moves down the diagonal gradient. Between the panels the head field shifts as the injection well (red) switches on in the transient period and the water table rises. Cells drawn with no color have drained below the plotted range.

#### Confirm the top layer went dry

Count how many top-layer cells have a head below the cell bottom (`bot[0]`, at 1590 m) after the first period — a head below the cell bottom means the cell is dry. This confirms that particles released into the top layer at the start of the simulation begin in dry cells.

In [ ]:
dry_layer0 = (hds_kper0[0] < bot[0]).sum()
print(
    f"Dry cells in top layer after first stress period: {dry_layer0} of {nrow * ncol}"
)

## Phase 2 — Build the particle-tracking model

The flow model is done and its heads and budgets are saved to disk. Now build a separate PRT model that reads those files and traces particles through the flow field the GWF model computed. PRT does not re-solve for flow — it consumes the GWF output.

Start by choosing where particles begin. Release them from cells in the top layer around the cell directly above the injection well. Each offset in the next cell is a zero-based `(dlayer, drow, dcolumn)` shift; here they pick the cell one layer above the well and its four edge neighbors.

In [ ]:
offsets = [(-1, 0, 0), (-1, -1, 0), (-1, 1, 0), (-1, 0, -1), (-1, 0, 1)]
release_cells = [
    (wel_cellid[0] + dk, wel_cellid[1] + di, wel_cellid[2] + dj)
    for dk, di, dj in offsets
]
release_cells

#### Turn release cells into release points

PRT needs the x, y, z coordinates of each particle, not just cell IDs. Rather than compute them by hand, reuse FloPy's MODPATH 7 (**MP7**) release-template utilities and convert the result to PRT's format. `LRCParticleData` fills the **layer-row-column bounding box** of the chosen cells — here a 3×3 block, 9 cells — with one particle each, and `.to_prp()` converts those into PRT release points on the flow model's grid.

In [ ]:
mp7_cell_data = flopy.modpath.CellDataType(
    rowcelldivisions=1, columncelldivisions=1, layercelldivisions=1
)
release_cells_T = np.array(release_cells).T
lrcregions = [
    [
        [
            min(release_cells_T[0]),
            min(release_cells_T[1]),
            min(release_cells_T[2]),
            max(release_cells_T[0]),
            max(release_cells_T[1]),
            max(release_cells_T[2]),
        ]
    ]
]
mp7_particle_data = flopy.modpath.LRCParticleData(
    subdivisiondata=mp7_cell_data,
    lrcregions=lrcregions,
)
mp7_pg = flopy.modpath.ParticleGroupLRCTemplate(
    particledata=mp7_particle_data,
)
release_pts = list(mp7_particle_data.to_prp(gwf.modelgrid))
release_pts

#### Assemble the PRT model

Build the PRT simulation the same way as the flow model, matching its temporal and spatial discretization exactly so the two grids line up. The PRT-specific packages are:

- **MIP** (Model Input Package, `ModflowPrtmip`) — the aquifer `porosity`, which converts cell fluxes to particle velocities.
- **PRP** (Particle Release Point package, `ModflowPrtprp`) — the release points from the previous step. `drape=True` handles a particle released into a dry cell by dropping it down to the first active cell beneath, i.e. onto the water table, instead of terminating it.
- **OC** (`ModflowPrtoc`) — write the particle tracks to `.trk` and `.csv` files.
- **FMI** (Flow Model Interface, `ModflowPrtfmi`) — the link to the flow model. It points at the GWF `.hds` and `.cbb` files (by relative path, hence the separate workspaces) and is how PRT reads the pre-computed flow field.
- **EMS** (`ModflowEms`) — the explicit solver PRT uses to advance particles.

The **drape-to-water-table** behavior is the key idea here: particles start in dry top-layer cells, and `drape` slides each one down to where the water actually is before tracking begins.

In [ ]:
import os

prt_name = f"{example_name}-prt"
prt_ws = base_ws / "prt"
prt_ws.mkdir(exist_ok=True, parents=True)

prt_sim = flopy.mf6.MFSimulation(
    sim_name=prt_name,
    exe_name="mf6",
    version="mf6",
    sim_ws=prt_ws,
)
prt_tdis = flopy.mf6.ModflowTdis(
    prt_sim,
    time_units=time_unit,
    nper=nper,
    perioddata=tdis_pd,
)
prt = flopy.mf6.ModflowPrt(
    prt_sim,
    modelname=prt_name,
    model_nam_file=f"{prt_name}.name",
)
prt_dis = flopy.mf6.ModflowPrtdis(
    prt,
    length_units=length_unit,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=delr,
    delc=delc,
    top=top,
    botm=bot,
)
prt_mip = flopy.mf6.ModflowPrtmip(prt, porosity=0.1)
prt_prp = flopy.mf6.ModflowPrtprp(
    prt,
    nreleasepts=len(release_pts),
    packagedata=release_pts,
    nreleasetimes=1,
    releasetimes=[(0.0,)],
    print_input=True,
    drape=True,
)
prt_oc = flopy.mf6.ModflowPrtoc(
    prt,
    budget_filerecord=[f"{prt_name}.bud"],
    track_filerecord=[f"{prt_name}.trk"],
    trackcsv_filerecord=[f"{prt_name}.csv"],
    saverecord=[("BUDGET", "ALL")],
    ntracktimes=1,
    tracktimes=[(100.0,)],
)
rel_gwf_ws = os.path.relpath(gwf_ws, prt_ws)
prt_fmi = flopy.mf6.ModflowPrtfmi(
    prt,
    packagedata=[
        ("GWFHEAD", f"{rel_gwf_ws}/{gwf_name}.hds"),
        ("GWFBUDGET", f"{rel_gwf_ws}/{gwf_name}.cbb"),
    ],
)
prt_ems = flopy.mf6.ModflowEms(prt_sim, pname="ems", filename=f"{prt_name}.ems")

#### Write and run the PRT model

Write and run the PRT model just like the flow model. Because FMI points at the flow output, this run traces particles through the heads and budgets the flow model already produced.

In [ ]:
prt_sim.write_simulation(silent=False)
success, buff = prt_sim.run_simulation(silent=False)
assert success, "MODFLOW 6 did not terminate normally"

#### Load and plot the pathlines

Read the particle track CSV into a pandas dataframe with `pd.read_csv()`. Draw the pathlines over the top-layer head with `PlotMapView.plot_pathline()`, and color each recorded particle position by its travel time `t` with a scatter plot.

In [ ]:
import pandas as pd

pls_prt = pd.read_csv(prt_ws / f"{prt_name}.csv")

with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    ax.set_aspect("equal")
    mm = flopy.plot.PlotMapView(gwf, ax=ax, layer=0)
    mm.plot_grid(alpha=0.2, lw=0.4)
    mm.plot_array(hds_kper1[1], alpha=0.25, vmin=1582, vmax=1587)
    mm.plot_bc("WELL", plotAll=True, kper=1, color="red")
    mm.plot_bc("CHD", plotAll=True, color="steelblue")
    mm.plot_pathline(pls_prt, layer="all", alpha=0.5, linewidth=0.5)
    pc = ax.scatter(pls_prt.x, pls_prt.y, c=pls_prt.t, s=5)
    cb = plt.colorbar(pc, shrink=0.5, pad=0.1)
    cb.ax.set_xlabel(r"Time (days)")
    ax.set_title("Pathlines, points colored by travel time", fontsize=11)
    ax.legend(
        handles=[
            Patch(color="red", label="WELL"),
            Patch(color="blue", label="CHD"),
        ],
    )

    fig.tight_layout()
    plt.show()

**What to look for.** Each line is one particle's path from its release cell in the top layer toward the constant-head outlet, and the scatter colors show travel time — darker points are earlier, lighter points later — so you can see where particles move quickly versus linger. Because `drape=True`, particles released into the dry top layer first drop onto the water table and then track laterally through the saturated cells; none are stranded in the dry region. The paths respond to the injection well (red), which adds water during the transient period.

#### Plot the pathlines in 3D

Render the same result in 3D with **PyVista**. Export the grid and pathlines to VTK meshes with FloPy's `Vtk` helper, add simple boxes to mark the well and constant-head cells, and color the particle tracks by travel time. The 3D view makes the drape-to-water-table behavior visible: particles drop from the dry top layer down onto the water table and then move laterally.

In [ ]:
import pyvista as pv
from flopy.export.vtk import Vtk


def get_nn(i, j):
    """Convert a structured grid 2D cell index to a node number."""
    return i * ncol + j


# create mesh for WELL boundary condition
wel_nn = get_nn(*wel_cellid[1:])
wel_pts = gwf.modelgrid.verts[gwf.modelgrid.iverts[wel_nn]]
wel_xs, wel_ys = list(zip(*wel_pts, strict=False))
wel_mesh = pv.Box(
    bounds=[
        min(wel_xs),
        max(wel_xs),
        min(wel_ys),
        max(wel_ys),
        bot[wel_cellid[0]].max(),
        bot[wel_cellid[0] - 1].max(),
    ]
)

# create mesh for CHD boundary condition
chd_nns = [get_nn(*chd0_cellid[1:]), get_nn(*chd1_cellid[1:])]
chd0_pts = gwf.modelgrid.verts[gwf.modelgrid.iverts[chd_nns[0]]]
chd1_pts = gwf.modelgrid.verts[gwf.modelgrid.iverts[chd_nns[1]]]
chd0_xs, chd0_ys = list(zip(*chd0_pts, strict=False))
chd1_xs, chd1_ys = list(zip(*chd1_pts, strict=False))
chd_mesh = pv.Box(
    bounds=[
        min(chd0_xs),
        max(chd0_xs),
        min(chd0_ys),
        max(chd0_ys),
        bot[chd0_cellid[0]].max(),
        bot[chd0_cellid[0] - 1].max(),
    ]
).merge(
    pv.Box(
        bounds=[
            min(chd1_xs),
            max(chd1_xs),
            min(chd1_ys),
            max(chd1_ys),
            bot[chd1_cellid[0]].max(),
            bot[chd1_cellid[0] - 1].max(),
        ]
    )
)

# create meshes for model results
vtk = Vtk(model=gwf, binary=False)
vtk.add_model(gwf)
vtk.add_pathline_points(pls_prt.to_records(index=False))
gwf_mesh, prt_mesh = vtk.to_pyvista()

# create plot
axes = pv.Axes(show_actor=False, actor_scale=2.0, line_width=5)
p = pv.Plotter(window_size=[700, 700])
p.enable_anti_aliasing()
p.add_mesh(gwf_mesh, opacity=0.1, style="wireframe")
p.add_mesh(wel_mesh, color="red", opacity=0.5)
p.add_mesh(chd_mesh, color="blue", opacity=0.5)
p.add_mesh(
    prt_mesh,
    point_size=8,
    line_width=2.5,
    smooth_shading=True,
    color="blue",
    scalars="t",
    scalar_bar_args={"title": "Time (days)"},
)
p.camera.elevation -= 25
p.show()

#### How PRT decides what to do in dry conditions

The behavior you configured with `drape` is one branch of two decision trees PRT walks for every particle. At **release** time (before tracking starts) PRT checks whether the release cell is active; if it is not and `drape` is on, PRT drapes the particle down to the first active cell — the water table — or terminates it if none is active. During **tracking**, if a particle enters a dry cell the `DRY_TRACKING_METHOD` option decides whether to stop it, drop it to the cell bottom or water table, or leave it stationary until the cell rewets. Change these settings and re-run to see how the pathlines respond.

### Release

```mermaid
flowchart LR
    ACTIVE --> |No| DRAPE(DRAPE)
    ACTIVE{Cell active?} --> |Yes| RELEASE[Release in specified cell]:::release
    DRAPE ==> |No| TERMINATE:::terminate
    DRAPE ==> |Yes| ACTIVE_UNDER{Active under?}
    ACTIVE_UNDER --> |Yes| RELEASE_AT_TABLE[Drape to active cell]:::release
    ACTIVE_UNDER --> |No| TERMINATE[Terminate]

    classDef release stroke:#98fb98
    classDef terminate stroke:#f08080
```

### Tracking

```mermaid
flowchart LR
    ACTIVE{Cell active?} --> |No| TERMINATE{Terminate}
    ACTIVE{Cell active?} --> |Yes| PARTICLE_DRY
    PARTICLE_DRY{Particle dry?} --> |Yes| DRY_TRACKING_METHOD(DRY_TRACKING_METHOD)
    DRY_TRACKING_METHOD ==> |STOP| TERMINATE[Terminate]:::terminate
    DRY_TRACKING_METHOD ==> |DROP| CELL_DRY{Cell dry?}
    CELL_DRY --> |Yes| DROP_BOTTOM[Pass to cell bottom]:::track
    CELL_DRY --> |No| DROP_TABLE([Pass to water table])
    DRY_TRACKING_METHOD ==> |STAY| STAY[Stationary]:::track
    DROP_TABLE --> TRACK:::track
    PARTICLE_DRY --> |No| TRACK[Track normally]

    classDef track stroke:#98fb98
    classDef terminate stroke:#f08080
```

## Recap

In this notebook you:

- Built a transient MODFLOW 6 **GWF** model whose upper layers dry out and re-wet, using convertible cells (NPF `icelltype=1`) and storage (STO).
- Saved the flow model's heads and cell budgets, and confirmed the top layer goes dry after the first stress period.
- Built a separate **PRT** model that consumes those saved files through the Flow Model Interface (FMI), released particles into the dry top layer, and used `drape=True` to drop them onto the water table.
- Plotted the resulting pathlines in map view (colored by travel time) and in 3D with PyVista.
- Reviewed the release and tracking decision trees that govern PRT's behavior in dry cells — the settings to revisit when you enable the Newton formulation on a second pass.